# Séance 11 · Exercices — Kaggle Titanic (2/2) : améliorer le score et éviter les pièges, sur un autre vaisseau · ⭐⭐⭐

**Niveau : ⭐⭐⭐ Avancé**

**Niveau de la séance : ⭐⭐⭐ Avancé** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

En séance, tu as comparé des modèles sur le Titanic, réglé leurs boutons et fait ta soumission finale sur Kaggle : **la soumission Kaggle, elle, reste dans la leçon**. Ici, mêmes outils, autres données : les passagers du *Spaceship Titanic*, que tu as préparés dans le carnet de la séance 10. Tu y verras aussi qu'un réflexe qui marchait sur le Titanic ne marche pas forcément partout.

**Comment travailler** : lis l'énoncé, code dans la cellule `# À toi`, lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
Tout tourne dans **Google Colab** (rien à installer). Exécute chaque cellule avec `Maj + Entrée`, dans l'ordre : les exercices réutilisent ce qui précède.

## Quiz d'ouverture (2 min)

Réponds dans ta tête, puis déplie la réponse. Ce sont les questions qu'on se pose à voix haute au début de la séance.

**1. Un binôme annonce 98 % sur le Titanic. Réaction ?**
- a. Champagne, c'est le meilleur modèle du monde
- b. Méfiance : c'est trop beau, il y a sûrement une fuite ou une erreur
- c. Normal pour une forêt aléatoire
- d. Il faut soumettre tout de suite

<details><summary>Réponse</summary>

**b. Méfiance : c'est trop beau, il y a sûrement une fuite ou une erreur** — Sur le Titanic, un bon score tourne autour de 0,77 à 0,80. Au-delà de 0,85, on cherche l'erreur.

</details>

## Préparation

Les 8 693 passagers du *Spaceship Titanic* (données fictives de Kaggle : en 2912, une anomalie de l'espace-temps a transporté près d'un passager sur deux dans une autre dimension), et la recette `preparer()` du carnet de la séance 10, déjà écrite. La cellule fabrique aussi `X` et `y`, un découpage `X_app` / `X_val` (25 % cachés), le faux test de l'exercice 12, le dictionnaire `modeles` des 4 familles, un tableau `X_suspect` pour l'exercice 9, et les fonctions `verifier()` (✅ / ❌) et `proche()` (compare deux nombres avec une tolérance).

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline

sns.set_theme(style="whitegrid")

# train.csv de la compétition Spaceship Titanic, deux copies publiques identiques, figées à une version
URLS = [
    "https://raw.githubusercontent.com/AmirMAAI/Spaceship_Titanic_dataset/a277ca40a73ffa2e83e477354cea20625e6470d2/spaceship-titanic_train.csv",
    "https://huggingface.co/datasets/Hugo0133/Spaceship-Titanic/resolve/dfc1e1d39a3193cfe99c53a51533b9ce04c71fb9/train.csv",
]


def charger(urls):
    """Essaie chaque copie l'une après l'autre ; s'arrête à la première qui répond."""
    for url in urls:
        try:
            return pd.read_csv(url)
        except Exception as erreur:
            print("Copie injoignable, on essaie la suivante :", erreur)
    raise RuntimeError("Aucune copie ne répond : vérifie la connexion internet, puis relance la cellule.")


vaisseau = charger(URLS)
print(vaisseau.shape[0], "passagers,", vaisseau.shape[1], "colonnes")

# --- La recette du carnet 10 : preparer() ---
DEPENSES = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
COLONNES = ["CryoSleep", "Age", "VIP", "Depense"] + DEPENSES + ["Planete", "Destination", "Pont", "Cote", "Taille_groupe"]

def preparer(df):
    """Transforme le tableau brut du vaisseau en tableau de nombres prêt pour un modèle."""
    d = df.copy()
    # 1. Les dépenses : une case vide compte 0 (la médiane de chaque colonne vaut 0)
    d[DEPENSES] = d[DEPENSES].fillna(0)
    d["Depense"] = d[DEPENSES].sum(axis=1)
    # 2. Les cases vides, avec un raisonnement
    d["CryoSleep"] = d["CryoSleep"].map({True: 1, False: 0}).fillna((d["Depense"] == 0).astype(int)).astype(int)
    d["VIP"] = d["VIP"].map({True: 1, False: 0}).fillna(0).astype(int)
    d["Age"] = d["Age"].fillna(d.groupby("HomePlanet")["Age"].transform("median")).fillna(27)
    # 3. Les nouvelles variables : la cabine (regex) et la taille du groupe
    cabine = d["Cabin"].str.extract(r"^(\w)/(\d+)/(\w)$")
    d["Pont"] = cabine[0].fillna("F").map(lambda pont: "ABCDEFGT".index(pont))
    d["Cote"] = (cabine[2] == "S").astype(int)
    groupe = d["PassengerId"].str.split("_").str[0]
    d["Taille_groupe"] = groupe.map(groupe.value_counts())
    # 4. Le texte en nombres
    d["Planete"] = d["HomePlanet"].fillna("Earth").map({"Earth": 0, "Europa": 1, "Mars": 2})
    d["Destination"] = d["Destination"].fillna("TRAPPIST-1e").map({"TRAPPIST-1e": 0, "55 Cancri e": 1, "PSO J318.5-22": 2})
    return d[COLONNES]


X = preparer(vaisseau)
y = vaisseau["Transported"].astype(int)
X_app, X_val, y_app, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

# --- Le faux test de l'exercice 12 : les groupes 8500 et plus, sans la réponse ---
numero_groupe = vaisseau["PassengerId"].str[:4].astype(int)
faux_test = vaisseau[numero_groupe >= 8500].drop(columns=["Transported"])
reponses_cachees = vaisseau.loc[numero_groupe >= 8500, "Transported"].astype(int)
vaisseau_reduit = vaisseau[numero_groupe < 8500]

# --- Les 4 familles de modèles de la leçon ---
modeles = {
    "Régression logistique": LogisticRegression(solver="liblinear"),
    "Arbre de décision": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Forêt aléatoire": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    "Gradient boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
}


# --- Pour l'exercice 9 : un tableau « suspect » avec 3 colonnes mystère (ne regarde pas comment il est fabriqué, joue le jeu !) ---
def _fabriquer_donnees_suspectes(X, y):
    rng = np.random.default_rng(0)
    d = X.copy()
    d["Code_A"] = rng.integers(0, 10, len(X))
    d["Code_B"] = np.where(rng.random(len(X)) < 0.9, y, 1 - y)
    d["Code_C"] = rng.normal(size=len(X)).round(2)
    return d


X_suspect = _fabriquer_donnees_suspectes(X, y)


# --- Vérification automatique : affiche ✅ ou ❌, ne plante jamais ---
def verifier(nom, condition):
    """condition : un booléen, ou une fonction sans argument qui renvoie un booléen."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))


def proche(valeur, attendu, tolerance=0.01):
    """Vrai si valeur est un nombre à moins de `tolerance` de attendu."""
    try:
        return abs(float(valeur) - attendu) <= tolerance
    except (TypeError, ValueError):
        return False

print("Prêt !", X.shape, "- preparer(), X, y, X_app, X_val, faux_test, modeles, X_suspect et verifier() sont chargés.")

## Exercice 1 ⭐ · Quatre modèles dans un tableau

Compare les 4 familles du dictionnaire `modeles` avec la **même** validation croisée (5 paquets) et range les résultats dans un DataFrame `tableau` à 3 colonnes : `Modèle`, `Exactitude moyenne`, `Écart-type`, trié de la meilleure exactitude à la moins bonne. `meilleur_modele` : le nom en première ligne.

Résultat attendu : 4 lignes en une vingtaine de secondes ; la forêt et le gradient boosting à égalité ou presque autour de `0.80`, l'arbre seul en dernier (≈ `0.77`).

<details><summary>Indice</summary>

Boucle `for nom, modele in modeles.items():`, `scores = cross_val_score(modele, X, y, cv=5)`, ajoute un dict `{"Modèle": nom, ...}` à une liste, puis `pd.DataFrame(liste).sort_values("Exactitude moyenne", ascending=False)`.

</details>

In [ ]:
# À toi
tableau = None
meilleur_modele = None

print(tableau)
print("Meilleur :", meilleur_modele)

In [ ]:
verifier("Exercice 1 · 4 lignes, 3 colonnes", lambda: tableau.shape == (4, 3) and list(tableau.columns) == ["Modèle", "Exactitude moyenne", "Écart-type"])
verifier("Exercice 1 · trié du meilleur au moins bon", lambda: tableau["Exactitude moyenne"].is_monotonic_decreasing)
verifier("Exercice 1 · forêt ≈ 0.80, arbre ≈ 0.77", lambda: proche(tableau.set_index("Modèle").loc["Forêt aléatoire", "Exactitude moyenne"], 0.7981, 0.005)
         and proche(tableau.set_index("Modèle").loc["Arbre de décision", "Exactitude moyenne"], 0.7705, 0.005))
verifier("Exercice 1 · meilleur_modele", lambda: meilleur_modele == tableau.iloc[0]["Modèle"])

<details><summary>Solution</summary>

```python
resultats = []
for nom, modele in modeles.items():
    scores = cross_val_score(modele, X, y, cv=5)
    resultats.append({"Modèle": nom, "Exactitude moyenne": scores.mean(), "Écart-type": scores.std()})
tableau = pd.DataFrame(resultats).sort_values("Exactitude moyenne", ascending=False).reset_index(drop=True)
meilleur_modele = tableau.iloc[0]["Modèle"]

print(tableau.round(4))
print("Meilleur :", meilleur_modele)
```

</details>

## Exercice 2 ⭐ · Stable ou capricieux ?

La moyenne ne dit pas tout : l'**écart-type** dit si le score change beaucoup d'un paquet à l'autre. À partir de `tableau` :
- `plus_stable` : le nom du modèle qui a le plus petit écart-type
- `scores_foret` : les 5 scores de la forêt (relance `cross_val_score` sur `modeles["Forêt aléatoire"]`)
- `etendue_foret` : son meilleur paquet moins son moins bon
- `ecart_meilleur_moins_bon` : la meilleure exactitude moyenne du tableau moins la moins bonne

Résultat attendu : la forêt varie de 3 points d'un paquet à l'autre (de `0.784` à `0.814`), alors que les 4 modèles tiennent dans moins de `0.03` : sur ces données, l'écart entre deux modèles est plus petit que le hasard du découpage.

<details><summary>Indice</summary>

`tableau.loc[tableau["Écart-type"].idxmin(), "Modèle"]` donne le plus stable. Pour l'étendue : `scores_foret.max() - scores_foret.min()`.

</details>

In [ ]:
# À toi
plus_stable = None
scores_foret = None
etendue_foret = None
ecart_meilleur_moins_bon = None

print("Plus stable :", plus_stable, "· étendue de la forêt :", etendue_foret, "· écart dans le tableau :", ecart_meilleur_moins_bon)

In [ ]:
verifier("Exercice 2 · plus_stable", lambda: plus_stable == tableau.loc[tableau["Écart-type"].idxmin(), "Modèle"])
verifier("Exercice 2 · étendue de la forêt ≈ 0.03", lambda: len(scores_foret) == 5 and proche(etendue_foret, 0.0298, 0.002))
verifier("Exercice 2 · les 4 modèles tiennent dans moins de 0.03", lambda: proche(ecart_meilleur_moins_bon, 0.0276, 0.003))

<details><summary>Solution</summary>

```python
plus_stable = tableau.loc[tableau["Écart-type"].idxmin(), "Modèle"]
scores_foret = cross_val_score(modeles["Forêt aléatoire"], X, y, cv=5)
etendue_foret = scores_foret.max() - scores_foret.min()
ecart_meilleur_moins_bon = tableau["Exactitude moyenne"].max() - tableau["Exactitude moyenne"].min()

print("Plus stable :", plus_stable, "· étendue de la forêt :", round(etendue_foret, 3), "· écart dans le tableau :", round(ecart_meilleur_moins_bon, 3))
print(scores_foret.round(3))
```

L'étendue de la forêt (0,030) dépasse l'écart entre le meilleur et le moins bon modèle (0,028) : un « meilleur » modèle à un demi-point près ne veut rien dire.

</details>

## Exercice 3 ⭐ · Qui décide ? L'importance des variables

Entraîne `foret = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)` sur `X, y`, puis :
- `importances` : une Series de `foret.feature_importances_`, indexée par `COLONNES`, triée de la plus grande à la plus petite
- `top2` : la liste des 2 variables les plus importantes
- `moins_utile` : le nom de la variable la moins utilisée

Trace un `barh`. Résultat attendu : les importances somment à `1`, et le duo de tête raconte la même histoire que les hypothèses du carnet 10.

<details><summary>Indice</summary>

`pd.Series(foret.feature_importances_, index=COLONNES).sort_values(ascending=False)`, puis `list(importances.index[:2])` et `importances.index[-1]`.

</details>

In [ ]:
# À toi
foret = None
importances = None
top2 = None
moins_utile = None

print(importances)
print("Top 2 :", top2, "· la moins utile :", moins_utile)

In [ ]:
verifier("Exercice 3 · les importances somment à 1", lambda: proche(importances.sum(), 1.0, 1e-6) and len(importances) == len(COLONNES))
verifier("Exercice 3 · triées", lambda: importances.is_monotonic_decreasing)
verifier("Exercice 3 · top2", lambda: set(top2) == {"Depense", "CryoSleep"})
verifier("Exercice 3 · moins_utile", lambda: moins_utile == "VIP")

<details><summary>Solution</summary>

```python
foret = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42).fit(X, y)
importances = pd.Series(foret.feature_importances_, index=COLONNES).sort_values(ascending=False)
top2 = list(importances.index[:2])
moins_utile = importances.index[-1]

print(importances.round(3))
print("Top 2 :", top2, "· la moins utile :", moins_utile)
importances.sort_values().plot(kind="barh", figsize=(6, 4), title="Qui décide ?")
plt.show()
# Depense et CryoSleep en tête : dormir et ne rien dépenser, c'est la même histoire racontée deux fois.
```

</details>

## Exercice 4 ⭐ · La règle bête à battre

Avant de se réjouir d'un score, on le compare aux règles les plus simples :
- `prediction_bete` : 1 pour les dormeurs, 0 pour les autres : c'est exactement la colonne `CryoSleep` de `X` (déjà remplie par `preparer()`)
- `exactitude_bete` : la part de passagers pour lesquels `prediction_bete == y`
- `exactitude_tous_transportes` : l'exactitude de la règle « tout le monde est transporté » (prédire 1 partout)

Résultat attendu : `0.72` pour « les dormeurs sont transportés », `0.50` pour « tout le monde est transporté ». La règle des dormeurs joue ici le rôle de « les femmes survivent » sur le Titanic : un modèle à 0.80 ne la bat que de 8 points.

<details><summary>Indice</summary>

`prediction_bete = X["CryoSleep"]`, puis `(prediction_bete == y).mean()`. « Tout le monde est transporté » a raison chaque fois que `y == 1`.

</details>

In [ ]:
# À toi
prediction_bete = None
exactitude_bete = None
exactitude_tous_transportes = None

print("Les dormeurs :", exactitude_bete, "- tout le monde :", exactitude_tous_transportes)

In [ ]:
verifier("Exercice 4 · règle des dormeurs ≈ 0.72", lambda: proche(exactitude_bete, 0.7233, 0.001))
verifier("Exercice 4 · tout le monde transporté ≈ 0.50", lambda: proche(exactitude_tous_transportes, 0.5036, 0.001))

<details><summary>Solution</summary>

```python
prediction_bete = X["CryoSleep"]
exactitude_bete = (prediction_bete == y).mean()
exactitude_tous_transportes = (y == 1).mean()

print("Les dormeurs :", round(exactitude_bete, 3), "- tout le monde :", round(exactitude_tous_transportes, 3))
```

</details>

## Exercice 5 ⭐⭐ · La courbe du par cœur

Fais grandir un arbre de décision de la profondeur 1 à 15 et mesure deux scores à chaque fois avec `cross_validate(..., cv=5, return_train_score=True)` :
- `profondeurs` : `range(1, 16)`
- `score_train` : la liste des 15 exactitudes moyennes sur les données vues (`cv["train_score"].mean()`)
- `score_valid` : la liste des 15 exactitudes moyennes sur les paquets cachés (`cv["test_score"].mean()`)
- `ecart_final` : `score_train[-1] - score_valid[-1]`, l'écart à la profondeur 15

Trace les deux courbes sur le même graphique. Résultat attendu : le train grimpe jusqu'à ≈ `0.93`, la validation plafonne autour de `0.785` puis redescend, `ecart_final` ≈ `0.17`.

<details><summary>Indice</summary>

`for p in profondeurs: cv = cross_validate(DecisionTreeClassifier(max_depth=p, random_state=42), X, y, cv=5, return_train_score=True)` puis `.append(...)` dans chaque liste. Graphique : `plt.plot(profondeurs, score_train, label="train")`, idem pour la validation, puis `plt.legend()`.

</details>

In [ ]:
# À toi
profondeurs = range(1, 16)
score_train = None
score_valid = None
ecart_final = None

print("Écart à la profondeur 15 :", ecart_final)

In [ ]:
verifier("Exercice 5 · 15 scores de chaque côté", lambda: len(score_train) == 15 and len(score_valid) == 15)
verifier("Exercice 5 · le train grimpe jusqu'à ≈ 0.93", lambda: proche(score_train[-1], 0.9259, 0.005) and score_train[-1] > score_train[0])
verifier("Exercice 5 · ecart_final ≈ 0.17", lambda: proche(ecart_final, 0.1675, 0.005))

<details><summary>Solution</summary>

```python
profondeurs = range(1, 16)
score_train, score_valid = [], []
for p in profondeurs:
    cv = cross_validate(DecisionTreeClassifier(max_depth=p, random_state=42), X, y, cv=5, return_train_score=True)
    score_train.append(cv["train_score"].mean())
    score_valid.append(cv["test_score"].mean())
ecart_final = score_train[-1] - score_valid[-1]

print("Écart à la profondeur 15 :", round(ecart_final, 3))
plt.plot(profondeurs, score_train, "o-", label="train (données vues)")
plt.plot(profondeurs, score_valid, "s-", label="validation (données cachées)")
plt.xlabel("max_depth")
plt.legend()
plt.show()
```

</details>

## Exercice 6 ⭐⭐ · La bonne profondeur

À partir des listes de l'exercice 5 :
- `meilleure_profondeur` : la profondeur (un entier entre 1 et 15) où `score_valid` est maximal
- `arbre_final` : un `DecisionTreeClassifier` de cette profondeur (`random_state=42`), entraîné sur `X, y`
- `nb_feuilles` : son nombre de feuilles (`arbre_final.get_n_leaves()`)

Résultat attendu : une profondeur de `6`, un arbre de 60 feuilles. À la profondeur 15, il en a plus de 1 000 : presque une feuille par poignée de passagers, c'est le par cœur.

<details><summary>Indice</summary>

`int(np.argmax(score_valid))` donne la position du maximum (à partir de 0) : la profondeur correspondante est `profondeurs[position]`.

</details>

In [ ]:
# À toi
meilleure_profondeur = None
arbre_final = None
nb_feuilles = None

print("Meilleure profondeur :", meilleure_profondeur, "·", nb_feuilles, "feuilles")

In [ ]:
verifier("Exercice 6 · meilleure profondeur", lambda: meilleure_profondeur == 6)
verifier("Exercice 6 · arbre_final entraîné à cette profondeur", lambda: arbre_final.get_depth() <= meilleure_profondeur and nb_feuilles == arbre_final.get_n_leaves())
verifier("Exercice 6 · 60 feuilles", lambda: nb_feuilles == 60)

<details><summary>Solution</summary>

```python
meilleure_profondeur = profondeurs[int(np.argmax(score_valid))]
arbre_final = DecisionTreeClassifier(max_depth=meilleure_profondeur, random_state=42).fit(X, y)
nb_feuilles = arbre_final.get_n_leaves()

print("Meilleure profondeur :", meilleure_profondeur, "·", nb_feuilles, "feuilles")
print("À la profondeur 15 :", DecisionTreeClassifier(max_depth=15, random_state=42).fit(X, y).get_n_leaves(), "feuilles")
```

</details>

## Exercice 7 ⭐⭐ · Un petit GridSearchCV

Laisse `GridSearchCV` tester toutes les combinaisons d'une petite grille sur une forêt :
- `grille = {"max_depth": [6, 8, 10, 12], "min_samples_leaf": [1, 5]}` (8 combinaisons)
- `recherche` : un `GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=42), grille, cv=5, n_jobs=-1)` entraîné sur `X, y`
- `meilleurs_reglages` : `recherche.best_params_`
- `meilleur_score` : `recherche.best_score_`

Résultat attendu : 8 × 5 = 40 entraînements en quelques secondes, un score ≈ `0.80`. Regarde aussi ce que le réglage gagne par rapport à la forêt du tableau de l'exercice 1.

<details><summary>Indice</summary>

`recherche.fit(X, y)` lance les 40 entraînements. `recherche.cv_results_["params"]` liste les combinaisons testées.

</details>

In [ ]:
# À toi
grille = {"max_depth": [6, 8, 10, 12], "min_samples_leaf": [1, 5]}
recherche = None
meilleurs_reglages = None
meilleur_score = None

print("Meilleurs réglages :", meilleurs_reglages, "· score :", meilleur_score)

In [ ]:
verifier("Exercice 7 · 8 combinaisons testées", lambda: len(recherche.cv_results_["params"]) == 8)
verifier("Exercice 7 · meilleurs réglages", lambda: meilleurs_reglages == {"max_depth": 12, "min_samples_leaf": 5})
verifier("Exercice 7 · score ≈ 0.80", lambda: proche(meilleur_score, 0.7987, 0.003))

<details><summary>Solution</summary>

```python
grille = {"max_depth": [6, 8, 10, 12], "min_samples_leaf": [1, 5]}
recherche = GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=42), grille, cv=5, n_jobs=-1)
recherche.fit(X, y)
meilleurs_reglages = recherche.best_params_
meilleur_score = recherche.best_score_

print("Meilleurs réglages :", meilleurs_reglages, "· score :", round(meilleur_score, 4))
# 0.7987 contre 0.7981 pour la forêt de l'exercice 1 : un réglage qui gagne moins que l'étendue entre deux paquets (exercice 2).
```

Sur ces données, régler les boutons ne rapporte presque rien : le gros du score vient des variables (le sommeil, les dépenses), pas des réglages. C'est fréquent, et c'est une bonne chose à dire dans une présentation.

</details>

## Exercice 8 ⭐⭐ · Le seuil de décision (predict_proba)

Le service de secours du vaisseau doit fouiller les cabines des transportés. Il préfère fouiller une cabine pour rien qu'oublier un transporté. Un modèle ne répond pas « oui / non » : il donne une **probabilité**, et `predict` la coupe à `0.5`. Ce seuil se choisit.

1. Entraîne une `RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)` sur `X_app, y_app`
2. `proba` : la probabilité d'être transporté de chaque passager de `X_val` (`predict_proba(X_val)[:, 1]`)
3. `nb_alertes_05` et `nb_alertes_03` : le nombre de passagers signalés avec un seuil de 0.5, puis de 0.3
4. `rappel_05` et `rappel_03` : parmi les **vrais** transportés de `y_val`, la part que chaque seuil attrape

Résultat attendu : `2174` probabilités, `1155` puis `1396` alertes, et un rappel qui **monte** quand le seuil baisse (≈ `0.82` → `0.90`) : on attrape plus de transportés, au prix de plus de cabines fouillées pour rien.

<details><summary>Indice</summary>

`(proba >= 0.3).astype(int)` donne les prédictions au seuil 0.3. Le rappel : `((prediction == 1) & (y_val == 1)).sum() / (y_val == 1).sum()`.

</details>

In [ ]:
# À toi
foret_seuil = None
proba = None
nb_alertes_05 = None
nb_alertes_03 = None
rappel_05 = None
rappel_03 = None

print("Seuil 0.5 :", nb_alertes_05, "alertes, rappel", rappel_05, "· seuil 0.3 :", nb_alertes_03, "alertes, rappel", rappel_03)

In [ ]:
verifier("Exercice 8 · une probabilité par passager caché", lambda: len(proba) == 2174 and 0 <= proba.min() and proba.max() <= 1)
verifier("Exercice 8 · nombre d'alertes", lambda: (nb_alertes_05, nb_alertes_03) == (1155, 1396))
verifier("Exercice 8 · rappel ≈ 0.82 puis ≈ 0.90", lambda: proche(rappel_05, 0.8214, 0.002) and proche(rappel_03, 0.9029, 0.002))

<details><summary>Solution</summary>

```python
foret_seuil = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42).fit(X_app, y_app)
proba = foret_seuil.predict_proba(X_val)[:, 1]
prediction_05 = (proba >= 0.5).astype(int)
prediction_03 = (proba >= 0.3).astype(int)
nb_alertes_05, nb_alertes_03 = prediction_05.sum(), prediction_03.sum()
rappel_05 = ((prediction_05 == 1) & (y_val == 1)).sum() / (y_val == 1).sum()
rappel_03 = ((prediction_03 == 1) & (y_val == 1)).sum() / (y_val == 1).sum()

print("Seuil 0.5 :", nb_alertes_05, "alertes, rappel", round(rappel_05, 3), "· seuil 0.3 :", nb_alertes_03, "alertes, rappel", round(rappel_03, 3))
# 241 cabines de plus à fouiller pour retrouver 89 transportés de plus (sur 1 092).
```

</details>

## Exercice 9 ⭐⭐⭐ · Détective : trouve la fuite de données

`X_suspect` (chargé dans la Préparation) contient les 14 colonnes de `X` **plus** 3 colonnes mystère `Code_A`, `Code_B`, `Code_C`, qui viendraient « d'un rapport retrouvé après l'accident ». L'une d'elles est une **fuite** : elle contient la réponse déguisée.
- `correlations` : la corrélation de chaque colonne de `X_suspect` avec `y` (une Series, triée)
- `colonne_fuite` : le nom de la colonne coupable
- `score_avec_fuite` : l'exactitude moyenne (cv=5) d'une `RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)` sur `X_suspect`
- `score_sans_fuite` : la même chose après avoir retiré la colonne coupable

Résultat attendu : une corrélation proche de 0.8 pour la coupable (`CryoSleep`, la vraie championne, plafonne à 0.47), et un score qui retombe de ≈ `0.92` à ≈ `0.80`.

<details><summary>Indice</summary>

`X_suspect.assign(Transported=y).corr()["Transported"].drop("Transported").sort_values()`. La coupable a une corrélation bien plus forte que n'importe quelle vraie variable : trop beau pour être vrai.

</details>

In [ ]:
# À toi
correlations = None
colonne_fuite = None
score_avec_fuite = None
score_sans_fuite = None

print("Coupable :", colonne_fuite, "· avec :", score_avec_fuite, "- sans :", score_sans_fuite)

In [ ]:
verifier("Exercice 9 · une corrélation par colonne", lambda: len(correlations) == X_suspect.shape[1])
verifier("Exercice 9 · la coupable", lambda: colonne_fuite == "Code_B")
verifier("Exercice 9 · score avec la fuite ≈ 0.92", lambda: proche(score_avec_fuite, 0.9184, 0.005))
verifier("Exercice 9 · score sans la fuite ≈ 0.80", lambda: proche(score_sans_fuite, 0.7956, 0.005))

<details><summary>Solution</summary>

```python
correlations = X_suspect.assign(Transported=y).corr()["Transported"].drop("Transported").sort_values()
colonne_fuite = correlations.abs().idxmax()
foret_detective = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
score_avec_fuite = cross_val_score(foret_detective, X_suspect, y, cv=5).mean()
score_sans_fuite = cross_val_score(foret_detective, X_suspect.drop(columns=[colonne_fuite]), y, cv=5).mean()

print(correlations.round(3))
print("Coupable :", colonne_fuite, "· avec :", round(score_avec_fuite, 3), "- sans :", round(score_sans_fuite, 3))
```

Code_B recopie la réponse 9 fois sur 10 : un rapport écrit **après** l'accident sait déjà qui a disparu. Sur un vrai `test.csv`, cette colonne n'existerait pas.

</details>

## Exercice 10 ⭐⭐⭐ · Un pipeline sklearn : la mise à l'échelle aide-t-elle toujours ?

Sur le Titanic, les k plus proches voisins avaient besoin d'une mise à l'échelle : `Fare` (jusqu'à 512) écrasait `Sex` (0 ou 1). Ici, `Depense` monte jusqu’à près de 36 000 et `VIP` vaut 0 ou 1. Le réflexe propre est un **pipeline** qui met à l'échelle puis classe, en une seule étape. Mesure s'il aide :
- `score_brut` : l'exactitude moyenne (cv=5) de `KNeighborsClassifier(n_neighbors=5)` sur `X` tel quel
- `pipeline` : `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`
- `score_pipeline` : son exactitude moyenne (cv=5)
- `l_echelle_aide` : `True` si le pipeline fait mieux que le modèle brut, sinon `False`

Résultat attendu : ≈ `0.76` brut, ≈ `0.76` avec le pipeline, à peine moins. Surprise : ici, la mise à l'échelle ne gagne rien.

<details><summary>Indice</summary>

`cross_val_score(pipeline, X, y, cv=5).mean()` : un pipeline se passe directement à `cross_val_score`, comme n'importe quel modèle.

</details>

In [ ]:
# À toi
score_brut = None
pipeline = None
score_pipeline = None
l_echelle_aide = None

print("k-NN brut :", score_brut, "- avec pipeline :", score_pipeline, "· l'échelle aide :", l_echelle_aide)

In [ ]:
verifier("Exercice 10 · pipeline = scaler + k-NN", lambda: isinstance(pipeline, Pipeline) and isinstance(pipeline.steps[0][1], StandardScaler) and isinstance(pipeline.steps[-1][1], KNeighborsClassifier))
verifier("Exercice 10 · score_brut ≈ 0.76", lambda: proche(score_brut, 0.7633, 0.005))
verifier("Exercice 10 · score_pipeline ≈ 0.76", lambda: proche(score_pipeline, 0.7564, 0.005))
verifier("Exercice 10 · l_echelle_aide", lambda: l_echelle_aide is False or l_echelle_aide == False)

<details><summary>Solution</summary>

```python
score_brut = cross_val_score(KNeighborsClassifier(n_neighbors=5), X, y, cv=5).mean()
pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
score_pipeline = cross_val_score(pipeline, X, y, cv=5).mean()
l_echelle_aide = bool(score_pipeline > score_brut)

print("k-NN brut :", round(score_brut, 3), "- avec pipeline :", round(score_pipeline, 3), "· l'échelle aide :", l_echelle_aide)
```

Pourquoi ? Sans mise à l'échelle, les distances ne voient presque que les dépenses... et les dépenses sont justement les variables les plus utiles (exercice 3). Mise à l'échelle, chaque colonne pèse autant : `VIP` ou `Destination`, qui n'apprennent presque rien au modèle, comptent soudain autant que les dépenses. Le pipeline reste la bonne façon de faire (le scaler est refait proprement dans chaque paquet de la validation croisée), mais un réflexe se vérifie avec un chiffre.

</details>

## Exercice 11 ⭐⭐⭐ · Score trop beau : trouve pourquoi

Un binôme annonce fièrement **97 %** avec la cellule ci-dessous. C'est trop beau. Lis le code, trouve l'erreur, et réponds :
- `raison` : `"A"`, `"B"` ou `"C"`
  - A : une colonne de `X` contient la réponse (fuite de données)
  - B : le modèle est noté sur les passagers qu'il a vus pendant l'entraînement
  - C : la forêt sans limite de profondeur est simplement excellente
- `score_honnete` : le score que ce même modèle obtient **vraiment** sur des passagers jamais vus
- `ecart` : `score_annonce - score_honnete`

Résultat attendu : un `score_honnete` autour de `0.77` et un écart de près de `0.20`.

<details><summary>Indice</summary>

Regarde sur quelles données `.score()` est appelé... et ce que contient `X_val`, qui n'est jamais utilisé.

</details>

In [ ]:
# À toi
modele_suspect = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_app, y_app)
score_annonce = modele_suspect.score(X_app, y_app)
print(f"Score annoncé : {score_annonce*100:.1f} %")

raison = None            # "A", "B" ou "C"
score_honnete = None
ecart = None

print("Raison :", raison, "- score honnête :", score_honnete, "- écart :", ecart)

In [ ]:
verifier("Exercice 11 · raison", raison == "B")
verifier("Exercice 11 · score_honnete ≈ 0.77 (mesuré sur X_val)", lambda: proche(score_honnete, 0.7732, 0.005))
verifier("Exercice 11 · ecart ≈ 0.19", lambda: proche(ecart, score_annonce - score_honnete, 1e-9) and ecart > 0.15)

<details><summary>Solution</summary>

```python
raison = "B"             # .score(X_app, y_app) : on corrige le contrôle avec le corrigé sous les yeux
score_honnete = modele_suspect.score(X_val, y_val)
ecart = score_annonce - score_honnete

print("Raison :", raison, "- score honnête :", round(score_honnete, 3), "- écart :", round(ecart, 3))
# Une forêt sans max_depth apprend les passagers vus par cœur (97 %) ; sur des passagers nouveaux, elle retombe à 77 %.
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi · la soumission finale, sans se tromper

Dernière ligne droite, comme sur Kaggle, avec le faux test de la Préparation (`faux_test` : 738 passagers sans la réponse, `reponses_cachees` : leurs vraies réponses, `vaisseau_reduit` : les 7 955 autres).
1. `modele_final` : un modèle de ton choix (le gradient boosting du tableau, la forêt réglée de l'exercice 7...), entraîné sur `preparer(vaisseau_reduit)` et `vaisseau_reduit["Transported"]`
2. `predictions` : ses prédictions sur `preparer(faux_test)`
3. écris `submission_finale_vaisseau.csv` : colonnes `PassengerId` et `Transported` (en `True` / `False`), `index=False`
4. `exactitude_finale` : la part de prédictions justes ; `exactitude_bete_faux_test` : celle de la règle « les dormeurs sont transportés » sur le même faux test

Résultat attendu : la règle bête fait `0.74` sur ce faux test ; ton modèle doit faire au moins `0.77`.

<details><summary>Indice</summary>

La règle bête sur le faux test : `preparer(faux_test)["CryoSleep"].values == reponses_cachees.values`, puis `.mean()`. Pour le fichier : `pd.DataFrame({"PassengerId": faux_test["PassengerId"], "Transported": predictions.astype(bool)})`.

</details>

In [ ]:
# À toi
modele_final = None
predictions = None
exactitude_finale = None
exactitude_bete_faux_test = None

print("Règle bête :", exactitude_bete_faux_test, "- ton modèle :", exactitude_finale)

In [ ]:
verifier("Exercice 12 · submission_finale_vaisseau.csv : 738 lignes, PassengerId et Transported",
         lambda: pd.read_csv("submission_finale_vaisseau.csv").shape == (738, 2) and list(pd.read_csv("submission_finale_vaisseau.csv").columns) == ["PassengerId", "Transported"])
verifier("Exercice 12 · les PassengerId sont ceux du faux test", lambda: list(pd.read_csv("submission_finale_vaisseau.csv")["PassengerId"]) == list(faux_test["PassengerId"]))
verifier("Exercice 12 · exactitude_bete_faux_test ≈ 0.74", lambda: proche(exactitude_bete_faux_test, 0.7425, 0.001))
verifier("Exercice 12 · ton modèle bat la règle bête (≥ 0.77)", lambda: proche(exactitude_finale, (pd.read_csv("submission_finale_vaisseau.csv")["Transported"].astype(int).values == reponses_cachees.values).mean(), 1e-9) and exactitude_finale >= 0.77)

<details><summary>Solution</summary>

```python
modele_final = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
modele_final.fit(preparer(vaisseau_reduit), vaisseau_reduit["Transported"].astype(int))
predictions = modele_final.predict(preparer(faux_test))

submission = pd.DataFrame({"PassengerId": faux_test["PassengerId"], "Transported": predictions.astype(bool)})
submission.to_csv("submission_finale_vaisseau.csv", index=False)

exactitude_finale = (predictions == reponses_cachees.values).mean()
exactitude_bete_faux_test = (preparer(faux_test)["CryoSleep"].values == reponses_cachees.values).mean()

print("Règle bête :", round(exactitude_bete_faux_test, 3), "- ton modèle :", round(exactitude_finale, 3))
# 0.80 contre 0.74 : 39 passagers de mieux sur 738.
```

</details>

## Bravo !

Tu sais comparer des modèles proprement, expliquer *pourquoi* l'un décide, repérer le par cœur sur une courbe, régler des boutons avec `GridSearchCV`, choisir un seuil selon ce qui coûte le plus cher, et surtout **te méfier** : d'un score trop beau, d'une fuite de données, et même d'un réflexe qui marchait ailleurs.

**Pour aller plus loin** : rejoins la compétition https://www.kaggle.com/competitions/spaceship-titanic, dépose son `test.csv` dans Colab, remplace `faux_test` par `pd.read_csv("test.csv")` dans l'exercice 12 (sans `reponses_cachees`, que Kaggle garde secrètes), et soumets `submission_finale_vaisseau.csv`.